# 은행 고객 이탈 예측

In [ ]:
import os
import pandas as pd
import numpy as np
from plt_rcs import *
import hds

In [ ]:
plt.rc(group='figure', figsize=(4, 4))

In [ ]:
os.getcwd()

In [ ]:
os.chdir('../../data')

In [ ]:
[i for i in os.listdir() if 'Bank' in i]

In [ ]:
objs = pd.read_pickle('Bank_Customer.pkl')

In [ ]:
globals().update(objs)

In [ ]:
%whos

In [ ]:
X_train, X_valid, y_train, y_valid = X_train, X_valid, y_train, y_valid

## 랜덤 포레스트 분류 모델 학습

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
model = RandomForestClassifier(random_state=0)

In [ ]:
model

In [ ]:
model.fit(X=X_train, y=y_train)

## 랜덤 포레스트 분류 모델 학습 결과 확인

In [ ]:
model.score(X=X_train, y=y_train)
# 0.9999166666666667
model.score(X=X_valid, y=y_valid)
# 0.9

## 특성중요도 확인

In [ ]:
pd.Series(
    data=model.feature_importances_, 
    index=model.feature_names_in_
).sort_values(ascending=False)
# Age        0.302313
# NumProd    0.167192
# Salary     0.126829
# Credit     0.122589
# Balance    0.087362
# Tenure     0.068599
# Active     0.039076
# Germany    0.029903
# Gender     0.024597
# HasCard    0.014111
# France     0.010037
# Spain      0.007391
# dtype: float64

In [ ]:
hds.plot.feature_importance(model)

## 분류 모델 성능 평가

In [ ]:
y_pred = model.predict(X=X_valid)

In [ ]:
hds.stat.clfmetrics(y_true=y_valid, y_pred=y_pred)

In [ ]:
y_prob = model.predict_proba(X=X_valid)

In [ ]:
hds.plot.roc_curve(y_true=y_valid, y_prob=y_prob)

In [ ]:
hds.plot.pr_curve(y_true=y_valid, y_prob=y_prob)

## 순열 중요도 확인

In [ ]:
from sklearn.inspection import permutation_importance

In [ ]:
# 순열 중요도 생성
pi = permutation_importance(
    estimator=model, X=X_valid, y=y_valid,
    random_state=0, scoring='roc_auc'
)

In [ ]:
# 특성별 순열 중요도의 평균 확인
pd.Series(
    data=pi['importances_mean'], index=model.feature_names_in_
).sort_values(ascending=False)
# NumProd    0.148834
# Age        0.107979
# Balance    0.023962
# Active     0.019315
# Germany    0.008270
# Gender     0.006819
# Credit     0.002010
# Tenure     0.001888
# Salary     0.000879
# Spain      0.000133
# France     0.000061
# HasCard   -0.000566
# dtype: float64

## 단변량 PDP 시각화

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

In [ ]:
PartialDependenceDisplay.from_estimator(
    estimator=model, X=X_valid,
    features=['Age', 'NumProd'],
    line_kw={'color': 'red', 'linewidth': 1.5}
)
plt.show()

## 단변량 PDP와 ICE 곡선 시각화

In [ ]:
PartialDependenceDisplay.from_estimator(
    estimator=model, X=X_valid,
    features=['Age', 'NumProd'],
    kind='both',
    random_state=0,
    pd_line_kw={'color': 'red', 'linewidth': 1.5},
    ice_lines_kw={'color': 'silver', 'alpha': 0.2}
)
plt.show()

In [ ]:
PartialDependenceDisplay.from_estimator(
    estimator=model, X=X_valid,
    features=['Tenure', 'Credit'],
    kind='both',
    random_state=0,
    pd_line_kw={'color': 'red', 'linewidth': 1.5},
    ice_lines_kw={'color': 'silver', 'alpha': 0.2}
)
plt.show()

## 이변량 PDP 시각화

In [ ]:
PartialDependenceDisplay.from_estimator(
    estimator=model, X=X_valid,
    features=[['Age', 'NumProd']],
    random_state=0,
    contour_kw={'cmap': 'Greys', 'alpha': 0.5}
)
plt.show()

In [ ]:
PartialDependenceDisplay.from_estimator(
    estimator=model, X=X_valid,
    features=[['Tenure', 'Credit']],
    random_state=0,
    contour_kw={'cmap': 'Greys', 'alpha': 0.5}
)
plt.show()

## LIME 설명기 생성

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

In [ ]:
lime_explainer = LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=model.feature_names_in_,
    mode='classification',
    class_names=['유지(0)', '이탈(1'],
    discretize_continuous=True,
    random_state=0
)

In [ ]:
index = np.argsort(y_prob[:,1])[-1]
index

In [ ]:
lime_exp = lime_explainer.explain_instance(
    data_row=X_valid.values[index],
    predict_fn=model.predict_proba,
    num_features=5
)

In [ ]:
from IPython.display import HTML, display

In [ ]:
lime_exp.save_to_file('LIME_Explanation.html')

In [ ]:
display(HTML('LIME_Explanation.html'))

In [ ]:
pd.Series(data=X_valid.values[index], index=model.feature_importances_)
# 0.122589       553.00
# 0.302313        60.00
# 0.068599         6.00
# 0.087362         0.00
# 0.167192         1.00
# 0.014111         1.00
# 0.039076         0.00
# 0.126829    143635.33
# 0.010037         1.00
# 0.029903         0.00
# 0.007391         0.00
# 0.024597         1.00
# dtype: float64

## SHAP

In [ ]:
import shap

In [ ]:
shap_explainer = shap.TreeExplainer(model=model)

In [ ]:
shap_values = shap_explainer.shap_values(X=X_valid, check_additivity=False)

## SHAP 값 확인

In [ ]:
shap_values.shape
# (3000, 12, 2)

In [ ]:
shap_1 = pd.DataFrame(data=shap_values[:, :, 1], columns=model.feature_names_in_)

In [ ]:
shap_1

In [ ]:
# SHAP 값의 절대값 평균 확인
shap_1.abs().mean().sort_values(ascending=False)
# Age        0.127524
# NumProd    0.122611
# Active     0.050641
# Gender     0.035896
# Germany    0.030100
# Balance    0.020121
# Credit     0.014884
# Salary     0.012301
# Tenure     0.008378
# France     0.007261
# Spain      0.004098
# HasCard    0.004063
# dtype: float64

## SHAP 특성 중요도 시각화

In [ ]:
shap.summary_plot(
    shap_values=shap_1.values,
    features=X_valid,
    plot_type='bar'
)

In [ ]:
shap.summary_plot(
    shap_values=shap_1.values,
    features=X_valid,
    plot_type='dot'
)

## SHAP 의존도 플랏 시각화

In [ ]:
shap.dependence_plot(
    ind='Age',
    shap_values=shap_1.values,
    features=X_valid,
    interaction_index=None
)

In [ ]:
shap.dependence_plot(
    ind='Age',
    shap_values=shap_1.values,
    features=X_valid,
    interaction_index='NumProd'
)

## 국소 샘플의 SHAP 기여도 확인

In [ ]:
base_value = shap_explainer.expected_value[1]
base_value
# np.float64(0.20317583333333328)

In [ ]:
# 검증셋에서 이탈 확률이 가장 높은 샘플의 SHAP값 확인
pd.Series(data=shap_1.values[index, :], index=model.feature_names_in_).sort_values(ascending=False)
# Age        0.382354
# NumProd    0.183389
# Active     0.096855
# Gender     0.057051
# Credit     0.045831
# Balance    0.036531
# Salary     0.014848
# Spain      0.003778
# HasCard    0.001516
# France    -0.003179
# Tenure    -0.006706
# Germany   -0.015444
# dtype: float64

## 국소 샘플의 SHAP 기여도 누적 플랏 시각화 결과

In [ ]:
shap.force_plot(
    base_value=base_value,
    shap_values=shap_1.values[index, :],
    features=X_valid.iloc[index, :],
    matplotlib=True
)